<a href="https://colab.research.google.com/github/eeeewyz/agent/blob/main/7_toolcall_reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graded Lab : Tool Use and Reflective Agents

In this lab, you will explore how AI agents can enhance research workflows by leveraging external tools and engaging in critical self-reflection. You'll learn how to build and integrate callable tools—such as web and academic search functions, and connect them to a language model using OpenAI's tool-calling API. Then, you’ll guide the agent to not only generate content but also **reflect** on its own output, improving the quality and depth of the final report. By the end of this lab, you will have implemented a mini agent capable of searching, reasoning, and publishing structured reports in HTML—laying the foundation for more advanced multi-step and autonomous AI systems.

### 🎯 Learning Objectives

By the end of this lab, you can:
- Chain steps into a research pipeline (**search → reflection → formatting**).
- Convert natural-language output into **styled HTML** suitable for sharing.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to write your solution code or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: red; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---

In [ ]:
# ================================
# Standard library imports
# ================================
import json

# ================================
# Third-party imports
# ================================
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

# ================================
# Local / project imports
# ================================
import research_tools

# ================================
# Environment setup
# ================================
load_dotenv()  # Load environment variables from .env file

# Instantiate OpenAI's client (you should use this in your graded functions)
CLIENT = OpenAI()

In [ ]:
import unittests

## Using Tools

You’ll use two research tools exposed in the `research_tools` module:
- **`arxiv_search_tool(query, max_results)`** – academic papers via arXiv API.
- **`tavily_search_tool(query, max_results, include_images)`** – general web search via Tavily.

Let's explore how the `arxiv_search_tool` works.

This tool searches arXiv and returns a list of papers with:
- `title`, `authors`, `published`, `summary`, `url`, and (if available) `link_pdf`.

Below, we run a quick test and print the results in a readable format. Next cell is editable so feel free to try some search queries:


In [ ]:
# Test the arXiv search tool
topic = "linear algebra"

arxiv_results = research_tools.arxiv_search_tool(topic, max_results=3)

# Show formatted arxiv_results
for i, paper in enumerate(arxiv_results, 1):
    if "error" in paper:
        print(f"❌ Error: {paper['error']}")
    else:
        print(f"📄 Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  Authors   : {', '.join(paper['authors'])}")
        print(f"  Published : {paper['published']}")
        print(f"  URL       : {paper['url']}\n")


print("\n🧾 Raw arxiv_Results:\n")
print(json.dumps(arxiv_results, indent=2))

📄 Paper 1
  Title     : Linear Mappings of Free Algebra
  Authors   : Aleks Kleyn
  Published : 2010-03-08
  URL       : http://arxiv.org/abs/1003.1544v2

📄 Paper 2
  Title     : Non-linear positive maps between $C^*$-algebras
  Authors   : Ali Dadkhah, Mox Sal Moslehian
  Published : 2018-11-07
  URL       : http://arxiv.org/abs/1811.03128v1

📄 Paper 3
  Title     : Grüss type inequalities for positive linear maps on $C^*$-algebras
  Authors   : Ali Dadkhah, Mohammad Sal Moslehian
  Published : 2016-10-12
  URL       : http://arxiv.org/abs/1610.03868v1


🧾 Raw arxiv_Results:

[
  {
    "title": "Linear Mappings of Free Algebra",
    "authors": [
      "Aleks Kleyn"
    ],
    "published": "2010-03-08",
    "url": "http://arxiv.org/abs/1003.1544v2",
    "summary": "For arbitrary F-algebra, in which the operation of addition is defined, I explore biring of matrices of mappings. The sum of matrices is determined by the sum in F-algebra, and the product of matrices is determined by the pr

The `tavily_search_tool` calls the Tavily API to fetch web results. Returns a list of dicts:
- `title`, `content`, `url` (and optional image URLs when `include_images=True`).

Run the cell to inspect sample output. Next cell is editable so feel free to try some search queries:

Tavily Search 本身不是 LLM，它更准确地说是一个给 AI / Agent 使用的搜索工具（Search API）。

In [ ]:
# Test the Tavily search tool
topic = "retrieval-augmented generation applications"

tavily_results = research_tools.tavily_search_tool(topic)
for item in tavily_results:
    print(item)

{'title': 'Retrieval-augmented generation - Wikipedia', 'content': 'Finally, the LLM can generate output based on both the query and the retrieved documents. Some models incorporate extra steps to improve output, such as the re-ranking of retrieved information, context selection, and fine-tuning "Fine-tuning (deep learning)").\n\n## Applications\n\n[edit]\n\nRetrieval-augmented generation is used in applications where generated responses need to be grounded in external or frequently updated information.[citation needed] [...] Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM\'s pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this 

## Tool Mapping

In the next cell you will define a dictionary that maps tool names (strings) to the actual Python functions. This allows the model to call tools by name during tool-calling. This dictionary will be used in your first graded function:

In [ ]:
# Tool mapping
TOOL_MAPPING = {
    "tavily_search_tool": research_tools.tavily_search_tool,
    "arxiv_search_tool": research_tools.arxiv_search_tool,
}

## Exercise 1: Generate Research Report with Tools
**Goal:** Implement `generate_research_report_with_tools(prompt)`.
In this exercise, you'll work on a function that generates a detailed research report with the assistance of online tools. Focus on setting up interaction with the language model and handling the responses effectively.

## Key Hints

### 1. Setting Up the Chat with the Language Model
- **Tool Selection**: Ensure that the tools are automatically selected by the model. Look into how to set `tool_choice` to "auto" within the function call. A helpful resource can be found in [OpenAI’s Function Calling Documentation](https://platform.openai.com/docs/guides/function-calling#tool-choice).
- **Parameter Configuration**: Consider the parameters already defined in your function, such as model, messages, and tools. Think about how these might be used in your setup.

### 2. Recording Tool Call Results
- **Understanding the `ChatCompletionMessage`** object will help you access the required attributes to save the messages. An example of `ChatCompletionMessage` looks like this:

```python
ChatCompletionMessage(
    content=None,
    refusal=None,
    role='assistant',
    annotations=[],
    audio=None,
    function_call=None,
    tool_calls=[
        ChatCompletionMessageFunctionToolCall(
            id='call_ymMki5TBB91efJhMPjgoqjop',
            function=Function(
                arguments='{"query":"radio observations of recurrent novae","max_results":5}',
                name='arxiv_search_tool'
            ),
            type='function'
        )
    ]
)
```
Assuming that `msg` if of type `ChatCompletionMessage`, if you wanted to get the `name` of a `tool_call` you can do something like:
```python
 for call in msg.tool_calls:
    tool_name = call.function.name
```
Finally, the `result` variable will be created by actually calling the function associated with each tool (`tool_func`).

By leveraging these hints, you'll work towards an implementation that enables robust data gathering and report generation through smart tool integration.

上面那段代码就是structured tool call

用户问题
   ↓
LLM
   ↓
产生 structured tool call
   ↓
{
  name: "arxiv_search_tool",
  arguments: {...}
}
   ↓
程序/runtime读取它
   ↓
真正执行 arxiv_search_tool(...)

ChatCompletionMessageFunctionToolCall(
    id='call_ymMki5TBB91efJhMPjgoqjop',
    function=Function(
        arguments='{"query":"radio observations of recurrent novae","max_results":5}',
        name='arxiv_search_tool'
    ),
    type='function'
)


msg = ChatCompletionMessage(
    content=None,
    role="assistant",
    tool_calls=[...]
)

那么 msg 是一个 ChatCompletionMessage 对象，所以可以：

msg.content
msg.role
msg.tool_calls

msg
= 整条 LLM message

    ↓ 里面有

msg.tool_calls
= 一个列表

    ↓ 列表里的每个元素

call
= 一次具体 tool call



for call in msg.tool_calls:

每次循环里，call 就等于其中一个这样的对象。

它内部有属性：

call.id
call.function
call.type

而 call.function 本身又是另一个对象：

Function(
    arguments='...',
    name='arxiv_search_tool'
)

所以它内部又可以继续点：

call.function.name
call.function.arguments

In [ ]:
# GRADED FUNCTION: generate_research_report_with_tools
def generate_research_report_with_tools(prompt: str, model: str = "gpt-4o") -> str:
    """
    使用 OpenAI 的 tool calling，
    让 LLM 可以调用 arXiv 和 Tavily 工具生成 research report。

    Args:
        prompt (str): 用户输入的问题
        model (str): 使用的 OpenAI 模型名称

    Returns:
        str: 最终生成的研究报告
    """

    # ---------------------------------------------------
    # 1. 初始化 conversation history
    # ---------------------------------------------------
    messages = [
        {
            "role": "system",

            # System Prompt：
            # 告诉 LLM 自己是什么角色、可以做什么、
            # 什么时候应该调用工具、最终答案应该是什么格式。
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "🔍 Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "📚 Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "🌐 When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "✍️ Use an academic tone, organize output into clearly labeled sections, and include "
                "inline citations or footnotes as needed.\n"
                "🚫 Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            )
        },

        # 用户真正提出的问题
        {
            "role": "user",
            "content": prompt
        }
    ]


    # ---------------------------------------------------
    # 2. 定义允许 LLM 使用的 tools
    # ---------------------------------------------------

    # 注意：
    # 这里不是实际执行工具的 Python function，
    # 而是 tool definition / schema。
    #
    # 它告诉 LLM：
    # 1. 有哪些工具
    # 2. 工具叫什么名字
    # 3. 每个工具需要什么参数
    #
    # LLM 根据这些 schema 决定要不要产生 structured tool call。
    tools = [
        research_tools.arxiv_tool_def,
        research_tools.tavily_tool_def
    ]


    # ---------------------------------------------------
    # 3. 设置最多允许进行多少轮 LLM <-> Tool 交互
    # ---------------------------------------------------

    # 防止 LLM 一直不断调用工具，形成无限循环
    max_turns = 10

    # 先初始化最终答案
    final_text = ""


    # ---------------------------------------------------
    # 4. 开始 Agent / Tool Calling Loop
    # ---------------------------------------------------

    for _ in range(max_turns):

        # ---------------------------------------------------
        # 4.1 调用 LLM
        # ---------------------------------------------------
        response = CLIENT.chat.completions.create(

            # 使用函数参数传进来的模型
            model=model,

            # 把当前完整 conversation history 给 LLM
            #
            # messages 会越来越长，例如：
            #
            # system
            # user
            # assistant(tool_call)
            # tool(result)
            # assistant(tool_call)
            # tool(result)
            # ...
            messages=messages,

            # 告诉模型现在有哪些工具可以使用
            tools=tools,

            # auto：
            # 让 LLM 自己判断：
            #
            # 1. 直接回答
            # 2. 调 arxiv
            # 3. 调 tavily
            # 4. 甚至调用多个工具
            tool_choice="auto",

            temperature=1,
        )


        # ---------------------------------------------------
        # 4.2 取出 LLM 返回的 assistant message
        # ---------------------------------------------------

        msg = response.choices[0].message


        # ---------------------------------------------------
        # 4.3 把 LLM 的输出保存进 messages
        # ---------------------------------------------------

        # 这一步非常重要。
        #
        # 如果 LLM 产生了 tool call，
        # 那么这个 tool call 也必须保存在 conversation history 中，
        # 后面 tool result 才能和它对应起来。
        messages.append(msg)


        # ---------------------------------------------------
        # 5. 判断 LLM 有没有产生 tool call
        # ---------------------------------------------------

        # 如果 msg.tool_calls 为空，
        # 说明 LLM 这次没有要求调用工具。
        #
        # 通常意味着：
        # 它已经拿到了足够的信息，
        # 现在正在返回最终答案。
        if not msg.tool_calls:

            final_text = msg.content

            print("✅ Final answer:")
            print(final_text)

            # 结束整个循环
            break


        # ---------------------------------------------------
        # 6. 如果有 tool_calls，就逐个执行
        # ---------------------------------------------------

        for call in msg.tool_calls:

            # ---------------------------------------------------
            # 6.1 读取 LLM 决定调用哪个工具
            # ---------------------------------------------------

            # 例如：
            #
            # "arxiv_search_tool"
            #
            # 或
            #
            # "tavily_search_tool"
            tool_name = call.function.name


            # ---------------------------------------------------
            # 6.2 读取 LLM 生成的参数
            # ---------------------------------------------------

            # call.function.arguments 通常是 JSON string：
            #
            # '{"query":"RAG applications","max_results":5}'
            #
            # json.loads 转成 Python dict：
            #
            # {
            #   "query": "RAG applications",
            #   "max_results": 5
            # }
            args = json.loads(call.function.arguments)


            # 打印出来只是方便开发者观察 Agent 在干什么
            print(f"🛠️ {tool_name}({args})")


            # ---------------------------------------------------
            # 6.3 真正执行 Python Tool
            # ---------------------------------------------------

            try:

                # TOOL_MAPPING 通常类似：
                #
                # TOOL_MAPPING = {
                #     "arxiv_search_tool":
                #         research_tools.arxiv_search_tool,
                #
                #     "tavily_search_tool":
                #         research_tools.tavily_search_tool
                # }
                #
                # 所以这里是：
                #
                # 工具名称
                #      ↓
                # 找到真正的 Python function
                tool_func = TOOL_MAPPING[tool_name]


                # ---------------------------------------------------
                # 真正调用工具
                # ---------------------------------------------------
                #
                # **args 相当于把字典展开
                #
                # 假设：
                #
                # args = {
                #     "query": "RAG",
                #     "max_results": 5
                # }
                #
                # 那么：
                #
                # tool_func(**args)
                #
                # 等价于：
                #
                # tool_func(
                #     query="RAG",
                #     max_results=5
                # )
                result = tool_func(**args)


            # 如果工具调用失败，
            # 也把 error 信息作为 result 返回给 LLM
            except Exception as e:

                result = {
                    "error": str(e)
                }


            # ---------------------------------------------------
            # 7. 把 Tool Result 包装成新的 message
            # ---------------------------------------------------

            new_msg = {

                # role="tool"
                #
                # 表明这条 message 不是 user，
                # 也不是 assistant，
                # 而是某个 tool 执行后的结果。
                "role": "tool",


                # ---------------------------------------------------
                # 对应之前 LLM 产生的那一个 tool call
                # ---------------------------------------------------
                #
                # 例如：
                #
                # call.id =
                # "call_ymMki5TBB91efJhMPjgoqjop"
                #
                # 这样 LLM / API 就知道：
                #
                # “这个 result 是对应哪一次 tool call 的”
                "tool_call_id": call.id,


                # 工具名称
                #
                # 例如：
                # "arxiv_search_tool"
                "name": tool_name,


                # ---------------------------------------------------
                # Tool 的真正执行结果
                # ---------------------------------------------------
                #
                # 因为 message content 一般需要 string，
                # 所以把 Python object 用 json.dumps 转成 JSON string。
                "content": json.dumps(result)
            }


            # ---------------------------------------------------
            # 8. 把工具结果加入 conversation history
            # ---------------------------------------------------

            messages.append(new_msg)


        # ---------------------------------------------------
        # 这里不会结束函数
        #
        # for loop 会回到上面，
        # 再次调用：
        #
        # CLIENT.chat.completions.create(...)
        #
        # 这一次 messages 中已经包含了工具结果，
        # 所以 LLM 可以看到搜索结果。
        # ---------------------------------------------------


    # ---------------------------------------------------
    # 9. 返回最终 research report
    # ---------------------------------------------------

    return final_text

msg = ChatCompletionMessage(
    content=None,
    role="assistant",
    tool_calls=[...]
)

那么 msg 是一个 ChatCompletionMessage 对象，所以可以：

msg.content
msg.role
msg.tool_calls

Run the following cell to check the correctness of your code. It might take a while so don't worry if it takes a couple of minutes to run:

In [ ]:
# Test your code!
unittests.test_generate_research_report_with_tools(generate_research_report_with_tools)

🛠️ arxiv_search_tool({'query': 'radio observations of recurrent novae', 'max_results': 5})
✅ Final answer:
### Overview of Radio Observations of Recurrent Novae

Radio observations of recurrent novae provide crucial insights into the physical processes governing these astronomical phenomena. Such observations can elucidate the mass ejection events, the evolution of the ejecta, and the associated shock conditions. Here, I will summarize findings from some notable research efforts documented in the scientific literature.

### Key Research Papers

1. **Lesson Learned from (some) Recurrent Novae**:
   - **Authors**: Elena Mason and Frederick M. Walters
   - **Publication Date**: March 2013
   - **Summary**: This study presents observations of the recurrent novae YY Dor and nova LMC 2009, focusing on early decline and nebular spectra. The analysis suggests that these novae, due to their spectral characteristics and evolutionary paths, likely originate from similar white dwarf progenitors an

## Exercise 2: Reflection + Rewrite

**Goal:** Implement `reflection_and_rewrite(report)`.

In this task, your goal is to develop a function that takes a report, analyzes it, generates a structured reflection, and produces an improved version of the report. This involves two main tasks: crafting a precise prompt and setting up a correctly configured response call to the language model.

## Key Steps

### 1. Create a User Prompt

- **Objective**: Guide the language model to output a structured response in JSON format.
- **Format**: Ensure the output includes two keys, `"reflection"` and `"revised_report"`.
- **Details**: Your reflection should cover strengths, limitations, suggestions, and opportunities. The revised report should incorporate these elements to improve clarity and academic tone.

### 2. Configure the Response Call

- **Parameters**: Use the specified model (e.g., `"gpt-4o-mini"`) and set the temperature equal to the `temperature` parameter of the graded function.
- **Structure**: Make sure the response setup directs the model properly, ensuring the JSON format is adhered to without additional commentary.


By implementing these steps, your function will effectively transform and improve the given reports. Handle JSON parsing carefully to ensure the output is valid and reliable. Happy coding!

In [ ]:
# GRADED FUNCTION: reflection_and_rewrite
def reflection_and_rewrite(
    report,
    model: str = "gpt-4o-mini",
    temperature: float = 0.3
) -> dict:
    """
    对研究报告进行反思并生成改进版本。

    Returns:
        {
            "reflection": "...",
            "revised_report": "..."
        }
    """

    # 统一处理输入格式
    report = research_tools.parse_input(report)

    ### START CODE HERE ###

    # 构造给 LLM 的提示词
    user_prompt = f"""
Analyze the following research report:

{report}

Provide a reflection covering:
- strengths
- limitations
- suggestions
- opportunities

Then provide a revised version of the report that improves clarity
and academic tone.

Return ONLY valid JSON in exactly this format:

{{
    "reflection": "<text>",
    "revised_report": "<text>"
}}

Do not include any additional commentary outside the JSON.
"""

    # 调用 LLM
    response = CLIENT.chat.completions.create(
        # 使用函数传入的模型
        model=model,

        messages=[
            # 定义 LLM 角色
            {
                "role": "system",
                "content": "You are an academic reviewer and editor."
            },

            # 传入上面构造好的 prompt
            {
                "role": "user",
                "content": user_prompt
            },
        ],

        # 使用函数传入的 temperature
        temperature=temperature
    )

    ### END CODE HERE ###

    # 取出 LLM 输出文本
    llm_output = response.choices[0].message.content.strip()

    # 把 JSON 字符串转换成 Python dict
    try:
        data = json.loads(llm_output)

    # 如果模型返回的不是合法 JSON，就报错
    except json.JSONDecodeError:
        raise Exception(
            "The output of the LLM was not valid JSON. Adjust your prompt."
        )

    # 返回需要的两个字段
    return {
        "reflection": str(data.get("reflection", "")).strip(),
        "revised_report": str(data.get("revised_report", "")).strip(),
    }

In [ ]:
# Test your code!
unittests.test_reflection_and_rewrite(reflection_and_rewrite)

 All tests passed!


## Exercise 3: Convert Report to HTML
**Goal:** Implement `convert_report_to_html(report)`.
This exercise focuses on transforming a plain text research report into a well-structured HTML document. You will build a function to facilitate this conversion using a language model.

## Key Steps

### 1. Create a User Prompt
- **Objective**: Instruct the model to transform plain text into HTML structure.
- **Format**: Ensure the output is valid, clean HTML with appropriate section headers, formatted paragraphs, and clickable links.
- **Details**: Preserve the citation style and request that the model responds only with HTML, without additional commentary.

### 2. Configure the Response Call
- **Parameters**: Use the specified model (e.g., `"gpt-4o"`) and set an appropriate temperature to balance creativity and accuracy.
- **Structure**: Configure the `CLIENT.chat.completions.create` call properly, using both system and user prompts to ensure a clear and focused task description.

By following these steps, you'll effectively convert plaintext reports into formatted HTML documents.

In [ ]:
# GRADED FUNCTION: convert_report_to_html
def convert_report_to_html(
    report,
    model: str = "gpt-4o",
    temperature: float = 0.5
) -> str:
    """
    将纯文本研究报告转换成 HTML 页面。

    参数：
        report:
            可以是纯文本，也可以是 tool-calling 阶段产生的 messages 列表。
        model:
            指定调用的 OpenAI 模型。
        temperature:
            控制生成结果的随机性，越低越稳定。

    返回：
        str：模型生成的 HTML 字符串。
    """

    # ① 统一输入格式
    # report 既可能是普通字符串，也可能是 messages 列表。
    # parse_input() 会自动识别并提取出真正需要转换的报告文本。
    report = research_tools.parse_input(report)

    # ② System Prompt
    # 告诉模型它的角色和核心任务：
    # 把普通文本报告转换成完整、干净的 HTML 文档。
    system_prompt = (
        "You convert plaintext reports into full clean HTML documents."
    )

    ### START CODE HERE ###

    # ③ 构造 User Prompt
    # 使用 f-string，把实际的 report 内容插入 prompt 中。
    # REPORT: 后面的内容就是需要被转换成 HTML 的原始报告。
    user_prompt = f"""
    Convert the plain text report to valid HTML.

    REPORT:
    {report}
    """

    # ④ 调用 OpenAI Chat Completions API
    response = CLIENT.chat.completions.create(

        # 使用函数传入的模型，例如 gpt-4o
        model=model,

        # messages 中包含：
        # system：定义模型角色
        # user：提供真正需要处理的报告
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],

        # 控制生成随机程度
        # 0 → 更稳定
        # 越高 → 输出变化越大
        temperature=temperature
    )

    ### END CODE HERE ###

    # ⑤ 获取模型生成的 HTML
    #
    # response
    #   └─ choices[0]
    #       └─ message
    #           └─ content
    #
    # .strip() 删除 HTML 前后的空格和换行。
    html = response.choices[0].message.content.strip()

    # ⑥ 返回最终 HTML 字符串
    return html

In [ ]:
# Test your code!
unittests.test_convert_report_to_html(convert_report_to_html)

 All tests passed!


### 🚀 End-to-End Pipeline

Run this cell to execute the full workflow:

1. Generate a research report (tools).
2. Reflect on the report.
3. Convert the report to HTML.

> You should see the rendered HTML below and two concise reflections in the console.

In [ ]:
# 1) Research with tools
prompt_ = "Radio observations of recurrent novae"
preliminary_report = generate_research_report_with_tools(prompt_)
print("=== Research Report (preliminary) ===\n")
print(preliminary_report)

# 2) Reflection on the report (use the final TEXT to avoid ambiguity)
reflection_text = reflection_and_rewrite(preliminary_report)   # <-- pass text, not messages
print("=== Reflection on Report ===\n")
print(reflection_text['reflection'], "\n")
print("=== Revised Report ===\n")
print(reflection_text['revised_report'], "\n")


# 3) Convert the report to HTML (use the TEXT and correct function name)
html = convert_report_to_html(reflection_text['revised_report'])

print("=== Generated HTML (preview) ===\n")
print((html or "")[:600], "\n... [truncated]\n")

# 4) Display full HTML
display(HTML(html))

🛠️ arxiv_search_tool({'query': 'radio observations of recurrent novae', 'max_results': 5})
✅ Final answer:
### Radio Observations of Recurrent Novae

#### Overview

Recurrent novae are a subclass of cataclysmic variable stars that experience multiple outbursts separated by decades. These events are caused by the accretion of material onto a white dwarf from a companion star until a thermonuclear explosion occurs. Observations across various wavelengths, including radio, are crucial for understanding the physical processes and properties of these systems.

#### Key Studies

1. **Lesson Learned from (some) Recurrent Novae**  
   - **Authors**: Elena Mason, Frederick M. Walters  
   - **Published**: March 12, 2013  
   - **Summary**: This study presents spectral data for recurrent novae such as YY Dor and nova LMC 2009, highlighting common spectral characteristics and post-outburst phases. The findings suggest similarities in the white dwarf progenitors involved in these events [arXiv lin

=== Generated HTML (preview) ===

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Radio Observations of Recurrent Novae</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            margin: 20px;
            padding: 0;
        }
        h1, h2, h3, h4 {
            color: #333;
        }
        a {
            color: #0066cc;
            text-decoration: none;
        }
        a:hover {
            text-decoration: underline;
        }
    </style>
</head 
... [truncated]



### 📌 “Expected Output” note (for the notebook text cell)

- `generate_research_report_with_tools` should return a **non-trivial string** (> 50 chars).

- `reflection_and_rewrite` should return a **dict** with **'reflection'** and **'revised\_report'** (both strings). The reflection should **mention** the four sections (Strengths, Limitations, Suggestions, Opportunities).

- `convert_report_to_html` should return a **string that looks like HTML** (e.g., includes `<html>`, `<h1>`, `<p>`, or closing tags).


---

## ✅ Wrap-Up

You built a mini research agent that can:
- 🔎 call tools (arXiv + Tavily),
- 🧠 reflect on its own output,
- 📰 publish a clean HTML report.

Great job!

### What to Submit
- Your notebook with Exercise 1–3 completed.

### Troubleshooting (quick)
- **Model/tool-call loop stalls?** Lower `max_turns` or print intermediate messages.
- **HTML looks odd?** Re-run conversion with a fresh assistant response.

**You’re done—nice work!** 🚀


## Check grading feedback

If you have collapsed the right panel to have more screen space for your code, as shown below:

<img src="./images/collapsed.png" alt="Collapsed Image" width="800" height="400"/>

You can click on the left-facing arrow button (highlighted in red) to view feedback for your submission after submitting it for grading. Once expanded, it should display like this:

<img src="./images/expanded.png" alt="Expanded Image" width="800" height="400"/>